# Monte Carlo Simulation of Mutual Fund NAV

Objective:
Project future NAV values for the next 5 years using historical return distributions.

Method:
- Historical daily returns
- Mean return (μ)
- Volatility (σ)
- 1000 random simulations
- 1260 trading days (252 × 5)

Outputs:
- Future NAV distribution
- Median NAV forecast
- 5th percentile NAV
- 95th percentile NAV

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
nav_df = pd.read_csv(
    "../data/processed/nav_history_clean.csv"
)

In [13]:
import pandas as pd
import numpy as np

# Ensure date format
nav_df["date"] = pd.to_datetime(nav_df["date"])

results = []

for code in nav_df["amfi_code"].unique():

    # Filter one fund
    fund_data = nav_df[
        nav_df["amfi_code"] == code
    ].copy()

    fund_data = fund_data.sort_values("date")

    # Skip funds with insufficient history
    if len(fund_data) < 30:
        continue

    # Daily returns
    fund_data["return"] = fund_data["nav"].pct_change()

    returns = fund_data["return"].dropna()

    # Mean return and volatility
    mu = returns.mean()
    sigma = returns.std()

    # Latest NAV
    current_nav = fund_data["nav"].iloc[-1]

    # Monte Carlo settings
    simulations = 1000
    trading_days = 252 * 5

    final_navs = []

    for sim in range(simulations):

        nav = current_nav

        for day in range(trading_days):

            daily_return = np.random.normal(mu, sigma)

            nav = nav * (1 + daily_return)

        final_navs.append(nav)

    # Forecast statistics
    median_nav = np.percentile(final_navs, 50)
    p5_nav = np.percentile(final_navs, 5)
    p95_nav = np.percentile(final_navs, 95)

    results.append({
        "amfi_code": code,
        "current_nav": round(current_nav, 2),
        "median_nav": round(median_nav, 2),
        "p5_nav": round(p5_nav, 2),
        "p95_nav": round(p95_nav, 2)
    })

# Create final dataframe
projection_df = pd.DataFrame(results)

# Show first few rows
print(projection_df.head())

# Save output
projection_df.to_csv(
    "../data/processed/monte_carlo_projection.csv",
    index=False
)

print("\nMonte Carlo projection saved successfully.")

   amfi_code  current_nav  median_nav   p5_nav  p95_nav
0     100016       583.61      656.94   384.36  1133.55
1     100025        31.88       39.38    34.18    45.46
2     100033       342.01     1198.87   613.35  2482.29
3     101206       773.29     2160.96  1273.47  3600.39
4     101207        53.98       77.48    32.38   209.81

Monte Carlo projection saved successfully.


In [14]:
projection_df.describe()

,amfi_code,current_nav,median_nav,p5_nav,p95_nav
count,40.000000,40.000000,40.000000,40.000000,40.000000
mean,120247.000000,357.612750,749.153500,486.965250,1241.939750
std,14534.998667,675.882091,1024.825359,922.335132,1352.640988
min,100016.000000,31.880000,39.380000,23.820000,45.460000
25%,118632.750000,103.545000,174.380000,95.642500,302.250000
50%,119551.500000,188.070000,495.850000,289.725000,719.490000
75%,120842.250000,381.375000,860.120000,494.730000,1451.115000
max,149324.000000,4268.550000,5893.930000,5784.240000,6011.600000
